---
title: "AI agent with Python and Local Gemini LLM"

description: "Building AI agent using llm"
author: "Aakash Basnet"
date: "2025/01/13"
page-layout: full
categories:
  - llm
  - gemini
  - AI
  - python
  - agent
format:
  html:
    code-fold: false
jupyter: python3
---


!["AI Agents (Generated by Imagen3)"](ai_agent.png)

# Installation
On your terminal run the following command to install Ollama:
```
pip install ollama
```

Pull and run the model of your choice. For this tutorial we are running Gemma2 2B model, since it is small and powerful:
```
ollama pull gemma2:2b
```

Install Langchain community module:
```
pip install langchain-ollama
```

Install yfinance to fetch stock data:
```
pip install yfinance
```

# Loading Model Locally

To use a large language model (LLM) like Gemma or Llama on your own machine, you need to run the model locally. This is made possible by tools like Ollama, which allow you to download, manage, and serve LLMs on your computer without needing cloud access.

**Steps to load a model locally with Ollama:**

1. **Install Ollama:**
   - Download and install Ollama from [https://ollama.com/](https://ollama.com/) for your operating system (macOS, Windows, or Linux).
   - Follow the installation instructions on the website.

2. **Pull a Model:**
   - Use the terminal to pull (download) a model. For example, to download the Gemma 1B model, run:
     ```bash
     ollama pull gemma3:1b
     ```
   - This command downloads the model weights and prepares them for use on your machine.

3. **Run the Model:**
   - Once pulled, Ollama automatically serves the model locally. You can now interact with it using the Ollama CLI or programmatically via an API (e.g., with LangChain).

4. **Integrate with Python (LangChain):**
   - In your Python code, you can use the `langchain-ollama` package to connect to the locally running Ollama server and use the model for inference.
   - Example:
     ```python
     from langchain_ollama.llms import OllamaLLM
     llm = OllamaLLM(model="gemma3:1b")
     ```

**Benefits of running models locally:**
- No data leaves your machine, ensuring privacy.
- No usage limits or API costs.
- Fast inference speeds (depending on your hardware).

**Note:**
- Running large models requires a modern CPU and enough RAM. Smaller models (like Gemma 2B) are suitable for most laptops and desktops.
- Make sure the Ollama server is running before you try to connect from Python.


In [1]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
import json
from pprint import pprint

llm = OllamaLLM(model="gemma3:1b",
                temperature = 0)

# Testing if llm load is successful
pprint(llm.invoke("Why color of sky is blue? Explain in 2 sentences"))


('The sky appears blue because of a phenomenon called Rayleigh scattering, '
 'where sunlight is scattered by tiny air molecules in the atmosphere. Blue '
 'light is scattered more than other colors, making it appear to fill the sky.')


# Example: Research Assistant Agent

Let's build a Research Assistant Agent step by step. This agent loads fundamental data for a stock using `yfinance` and then uses an LLM to analyze and summarize the findings.

### Step 1: Fetch Fundamental Data with yfinance

Next, we use the `yfinance` library to fetch fundamental data for a stock (e.g., Apple). This includes company info, financials, balance sheet, and cash flow.


In [2]:

import yfinance as yf

# Load fundamental data for a stock (e.g., Apple)
ticker = yf.Ticker("AAPL")

# Get basic info and financials
data = {
    "info": ticker.info,
    "financials": ticker.financials.to_dict(),
    "balance_sheet": ticker.balance_sheet.to_dict(),
    "cashflow": ticker.cashflow.to_dict(),
}

pprint(data["info"])


{'52WeekChange': 0.04945779,
 'SandP52WeekChange': 0.13881528,
 'address1': 'One Apple Park Way',
 'allTimeHigh': 288.62,
 'allTimeLow': 0.049107,
 'ask': 274.97,
 'askSize': 4,
 'auditRisk': 7,
 'averageAnalystRating': '2.0 - Buy',
 'averageDailyVolume10Day': 50896430,
 'averageDailyVolume3Month': 47275577,
 'averageVolume': 47275577,
 'averageVolume10days': 50896430,
 'beta': 1.107,
 'bid': 272.0,
 'bidSize': 1,
 'boardRisk': 1,
 'bookValue': 4.991,
 'city': 'Cupertino',
 'companyOfficers': [{'age': 63,
                      'exercisedValue': 0,
                      'fiscalYear': 2024,
                      'maxAge': 1,
                      'name': 'Mr. Timothy D. Cook',
                      'title': 'CEO & Director',
                      'totalPay': 16520856,
                      'unexercisedValue': 0,
                      'yearBorn': 1961},
                     {'age': 60,
                      'exercisedValue': 0,
                      'fiscalYear': 2024,
                   

### Step 2: Load the LLM Model

First, we load the local LLM model using LangChain and Ollama. This sets up the language model that will be used for all downstream tasks.

In [3]:
llm = OllamaLLM(model="gemma3:1b",
                temperature = 0)

### Step 3: Create a High-Quality Prompt

A well-crafted prompt is crucial for getting useful and reliable results from an LLM. Here, we define the agent's role, tone, and the steps it should follow. The prompt instructs the LLM to act as a financial research assistant, respond in a professional and concise tone, and structure the output clearly.

**Prompt Design:**
- **Role:** Financial research assistant
- **Tone:** Professional, concise, and objective
- **Instructions:**
    1. Summarize the company's business in 2-3 sentences.
    2. Highlight recent financial performance (growth, profitability, etc.).
    3. List notable strengths and risks as bullet points.
    4. Use only the provided data; do not speculate.
    5. Output should be clear and easy to read.

Let's define this prompt in code.

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a financial research assistant. Respond in a professional, concise, and objective tone.
        Your task is to analyze the provided company fundamental data and:
        1. Summarize the company's business in 2-3 sentences.
        2. Highlight recent financial performance (growth, profitability, etc.).
        3. List notable strengths and risks as bullet points.
        4. Use only the provided data; do not speculate or add external information.
        6. Output must be in plain text (no markdown, no JSON, no code blocks).
        7. Do not include any disclaimers or extra commentary.
        
     Data will be provided as JSON.
    """),
    ("human", """
        {fundamental_json}
    """),
])


### Step 5: Create and Run the Agent

Now, we combine the LLM, the prompt, and the data to create the agent. The agent streams the LLM's output as it analyzes the data and prints the summary in plain text format.

In [13]:
chain = prompt | llm

user_data = {"fundamental_json": json.dumps(data["info"])}

result = ""
async for chunk in chain.astream(user_data):
    result += chunk
    print(chunk, end="", flush=True)


The Apple Apple Inc. Inc. (AAPL (AAPL) presents) presents a business a business model centered model centered around the around the design, design, manufacture, manufacture, and marketing and marketing of consumer of consumer electronics, electronics, including smartphones including smartphones, computers, computers, tablets, tablets, wearables, wearables, and, and accessories. accessories. It operates It operates through various through various platforms, platforms, including the including the App Store App Store, offering, offering a wide a wide range of range of digital content digital content and services and services. The. The company’ company’s financials financial performance is performance is characterized by characterized by consistent revenue consistent revenue growth, growth, profitability, profitability, and a and a strong brand strong brand presence. presence. Notable strengths Notable strengths include a include a loyal customer loyal customer base, base, a diversified a 